In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay

TASK 1: LOAD & DESCRIBE DATA

In [ ]:
# Download date: November 16, 2026
df = pd.read_csv("/Users/ajsaamirkyzy/Downloads/Food_Inspections_20261116.csv", nrows=25000)
print(f"Raw shape: {df.shape}")

Raw shape: (25000, 17)


In [ ]:
# Filter for binary outcomes(1 = Pass, 0 = Fail)
df = df[df['Results'].isin(['Pass', 'Fail'])].copy()
df['y'] = (df['Results'] == 'Pass').astype(int)  # 1 = Pass, 0 = Fail

In [ ]:
# Engineer meaningful features from raw text and timestamp columns
df['num_violations'] = df['Violations'].fillna('').apply(
    lambda x: 0 if x == '' else x.count('|') + 1
)
df['has_critical'] = df['Violations'].fillna('').str.contains('CRITICAL', case=False).astype(int)
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'], errors='coerce')
df['insp_month'] = df['Inspection Date'].dt.month

In [ ]:
# Group rare facility types into an 'Other' category
top_facilities = df['Facility Type'].value_counts().nlargest(5).index
df['Facility_Clean'] = np.where(df['Facility Type'].isin(top_facilities), df['Facility Type'], 'Other')

TASK 2: CLEAN THE DATA

In [7]:
features = ['num_violations', 'has_critical', 'insp_month', 'Latitude', 'Longitude', 'Risk', 'Facility_Clean']
df_clean = df[features + ['y']].copy()

In [ ]:
# Remove duplicate rows and missing values
before_clean = len(df_clean)
df_clean = df_clean.drop_duplicates().dropna()
print(f"Rows before cleaning: {before_clean}, after removing duplicates & NaNs: {len(df_clean)}")

Rows before cleaning: 17782, after removing duplicates & NaNs: 16935


In [ ]:
numeric_cols = ['num_violations', 'has_critical', 'insp_month', 'Latitude', 'Longitude']
for col in numeric_cols:
    if col in df_clean.columns:
        if df_clean[col].dtype == object:
            df_clean[col] = df_clean[col].astype(str).str.replace(',', '.').astype(float)
        else:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')


df_clean = df_clean.dropna()

In [13]:
counts = df_clean['y'].value_counts()
props = df_clean['y'].value_counts(normalize=True) * 100
print("\n--- TASK 1: CLASS BALANCE ---")
print(f"Pass (1): {counts[1]} ({props[1]:.2f}%)")
print(f"Fail (0): {counts[0]} ({props[0]:.2f}%)")


--- TASK 1: CLASS BALANCE ---
Pass (1): 12633 (74.60%)
Fail (0): 4302 (25.40%)


In [ ]:
# One-hot encoding
df_encoded = pd.get_dummies(df_clean, columns=['Risk', 'Facility_Clean'], drop_first=True, dtype=int)

X = df_encoded.drop(columns=['y'])
y = df_encoded['y']
print(f"Total features after dummy encoding: {X.shape[1]}")

Total features after dummy encoding: 13


TASK 3: SPLIT & SCALE

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

TASK 4: SIMPLE MODEL (2 MOST IMPORTANT FEATURES)

In [16]:
top_2 = ['num_violations', 'has_critical']
simple_model = LogisticRegression(random_state=42)
simple_model.fit(X_train_scaled[top_2], y_train)

print("\n--- TASK 4: SIMPLE MODEL ---")
print(f"Intercept (beta_0): {simple_model.intercept_[0]:.4f}")
for feat, coef in zip(top_2, simple_model.coef_[0]):
    print(f"{feat:16s} | Beta = {coef:8.4f} | Odds Ratio exp(Beta) = {np.exp(coef):8.4f}")


--- TASK 4: SIMPLE MODEL ---
Intercept (beta_0): 1.3253
num_violations   | Beta =  -1.5374 | Odds Ratio exp(Beta) =   0.2149
has_critical     | Beta =  -0.0471 | Odds Ratio exp(Beta) =   0.9540


TASK 5: FULL MODEL

In [ ]:
full_model = LogisticRegression(max_iter=1000, random_state=42)
full_model.fit(X_train_scaled, y_train)

# predictions for the first 5 test samples
print("\n--- TASK 5: 5 TEST PREDICTIONS ---")
sample_5_X = X_test_scaled.iloc[:5]
sample_5_y = y_test.iloc[:5].values
sample_5_prob = full_model.predict_proba(sample_5_X)[:, 1]
sample_5_pred = (sample_5_prob >= 0.5).astype(int)

pred_table = pd.DataFrame({
    'Row_ID': X_test.index[:5],
    'True_y': sample_5_y,
    'Prob_y1': np.round(sample_5_prob, 4),
    'Pred_y': sample_5_pred
})
print(pred_table.to_string(index=False))
# Evaluate full model performance on test set
y_test_pred = full_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_test_pred)
acc = accuracy_score(y_test, y_test_pred)

print(f"\nFull Model Test Accuracy: {acc:.4f}")
print("Confusion Matrix:\n", cm)


--- TASK 5: 5 TEST PREDICTIONS ---
 Row_ID  True_y  Prob_y1  Pred_y
  17570       1   0.9253       1
  11491       1   0.7688       1
   8239       1   0.8925       1
    793       1   0.8032       1
  19962       1   0.8252       1

Full Model Test Accuracy: 0.8172
Confusion Matrix:
 [[ 572  719]
 [ 210 3580]]


TASK 6: REGULARIZATION (C = 0.01, 1, 100)

In [19]:
print("\n--- TASK 6: REGULARIZATION ---")
for c in [0.01, 1, 100]:
    clf = LogisticRegression(C=c, max_iter=1000, random_state=42)
    clf.fit(X_train_scaled, y_train)
    acc_c = accuracy_score(y_test, clf.predict(X_test_scaled))
    l2_norm = np.linalg.norm(clf.coef_[0])
    max_coef = np.max(np.abs(clf.coef_[0]))
    print(f"C = {c:<5} | Test Acc = {acc_c:.4f} | L2 Norm = {l2_norm:.4f} | Max |Beta| = {max_coef:.4f}")


--- TASK 6: REGULARIZATION ---
C = 0.01  | Test Acc = 0.8170 | L2 Norm = 1.5214 | Max |Beta| = 1.4832
C = 1     | Test Acc = 0.8172 | L2 Norm = 1.7300 | Max |Beta| = 1.6634
C = 100   | Test Acc = 0.8172 | L2 Norm = 1.7329 | Max |Beta| = 1.6656
